<a href="https://colab.research.google.com/github/havvayamanofficial/Techistanbul_AI_PythonProject/blob/master/%C3%96dev__3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# =============================================================================
# YAPAY ZEKA BOOTCAMP - ÖDEV #3: BELEDİYE MESAJ YÖNLENDİRME HATTI (REVİZE)
# =============================================================================

# 1. Gerekli Kütüphanelerin Kurulması
print("[INFO] Kütüphaneler kuruluyor, lütfen bekleyin...")
!pip install -q transformers sentence-transformers colorama reportlab

import io
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer
from sentence_transformers import SentenceTransformer, util
from sklearn.model_selection import train_test_split
import keras

# -------------------------------------------------------------------------
# VERİ SETİNDEKİ TIRNAK İŞARETLERİNDEN ARINDIRILMIŞ GÜVENLİ METİN
# -------------------------------------------------------------------------
raw_data = """mesaj,departman
Sokak aralarında çöp yığınları oluştu.,Temizlik ve Çöp
Birisi boş arsaya moloz dökmüş.,Temizlik ve Çöp
Çimlere ilaçlama ne zaman yapılacak?,Park ve Bahçeler
Kafeterya hijyen kurallarına uymuyor.,Zabıta
Otobüs güzergâhı neden değişti?,Ulaşım ve Trafik
Lağım kokusu bütün sokağı sardı.,Su ve Kanalizasyon
İş yeri tabelası yaya yolunu kapatıyor.,Zabıta
Evimizin önüne fidan dikmek istiyoruz.,Park ve Bahçeler
Komşu iş yeri çok yüksek sesle müzik çalıyor.,Zabıta
Sokağımıza yeni bir çöp konteyneri konulmasını istiyorum.,Temizlik ve Çöp
Gece boyunca borulardan tıkırtı sesi geliyor ve su kesik kesik akıyor.,Su ve Kanalizasyon
42 numaralı otobüs yarım saattir gelmiyor.,Ulaşım ve Trafik
Minibüsler çok kalabalık sefer sayısı artırılmalı.,Ulaşım ve Trafik
Yeni aldığım daireye su aboneliği açtırmak istiyorum.,Su ve Kanalizasyon
Terk edilmiş bir araç aylardır sokakta duruyor.,Zabıta
Otobüs şoförü durakta durmadan geçti.,Ulaşım ve Trafik
Mazgallar tıkalı yağmur suyu yolda birikiyor.,Su ve Kanalizasyon
Parktaki aydınlatmalar yanmıyor akşam karanlık oluyor.,Park ve Bahçeler
Çöpler üç gündür toplanmadı.,Temizlik ve Çöp
Yol çalışmasında su borusu zarar görmüş.,Su ve Kanalizasyon
Asfalt çalışması ne zaman bitecek?,Ulaşım ve Trafik
Musluktan çamurlu su geliyor.,Su ve Kanalizasyon
Apartmanın ana gider hattı tıkanmış.,Su ve Kanalizasyon
Bir restoranda fiyat listesi asılı değil.,Zabıta
Evin önündeki kanalizasyon kapağı yerinden çıkmış.,Su ve Kanalizasyon
Gıda satan bir büfede soğuk zincire uyulmuyor.,Zabıta
Durağın tabelası yıkılmış.,Ulaşım ve Trafik
Atık pil toplama kutusu nerede var?,Temizlik ve Çöp
Parkın içindeki yürüyüş yolu bozuk.,Park ve Bahçeler
Pastane etiketteki fiyattan fazla ücret aldı.,Zabıta
Mahallemize yeni bir park yapılmasını istiyoruz.,Park ve Bahçeler
Engelli rampası olmayan otobüsler hâlâ çalışıyor.,Ulaşım ve Trafik
Çöp kamyonu sabahın beşinde çok gürültü yapıyor.,Temizlik ve Çöp
Gece otobüs seferi eklenmesini talep ediyorum.,Ulaşım ve Trafik
Apartmanın önüne bırakılan çöp poşetlerini kimse almıyor.,Temizlik ve Çöp
Yolda büyük bir çukur var araçlar zarar görüyor.,Ulaşım ve Trafik
Çöp konteynerleri yolun ortasına sürüklenmiş.,Temizlik ve Çöp
Konteynerin kapağı kırık kediler çöpleri etrafa saçıyor.,Temizlik ve Çöp
Çöp toplama saatleri değişti mi?,Temizlik ve Çöp
Refüjdeki çiçekler kurumuş.,Park ve Bahçeler
Mahalledeki parkın çimleri hiç biçilmiyor.,Park ve Bahçeler
Mahallede izinsiz hurdacılık yapılıyor.,Zabıta
Ağaçların budanma zamanı geldi dallar camlarımıza vuruyor.,Park ve Bahçeler
Tramvay hattında arıza var seferler durdu.,Ulaşım ve Trafik
Kavşaktaki trafik ışığı bozuk sürekli kırmızı yanıyor.,Ulaşım ve Trafik
Su sayacım bozuk sürekli dönüyor.,Su ve Kanalizasyon
Köpek parkı yapılması mümkün mü?,Park ve Bahçeler
Bir dükkân kapısının önüne sürekli mal yığıyor.,Zabıta
Cadde süpürülmüyor yapraklar birikmiş.,Temizlik ve Çöp
Kaldırımı işgal eden seyyar satıcılar yürümeyi engelliyor.,Zabıta
Konteyner çok kötü kokuyor yıkanması gerekiyor.,Temizlik ve Çöp
Kaldırımda kırık cam şişeler var çocuklar için tehlikeli.,Temizlik ve Çöp
Planlı bir kesinti olacaksa önceden haber verebilir misiniz?,Su ve Kanalizasyon
Mahalledeki market son kullanma tarihi geçmiş ürün satıyor.,Zabıta
Sokağımızdaki rögar taşmış her yer pis su oldu.,Su ve Kanalizasyon
Fırın ekmeği eksik gramajla satıyor.,Zabıta
Parkın çevre çiti kırılmış çocuklar yola çıkabiliyor.,Park ve Bahçeler
Ruhsatsız bir iş yeri açılmış.,Zabıta
Su faturam bu ay çok yüksek geldi.,Su ve Kanalizasyon
Bayram sonrası kurban atıkları sokağa bırakılmış.,Temizlik ve Çöp
Kasap etiketsiz et satıyor.,Zabıta
Çeşmeden gelen suyun tadı ve kokusu değişti.,Su ve Kanalizasyon
Sokağımıza hız kesici konulabilir mi?,Ulaşım ve Trafik
Kaldırımın altından su fışkırıyor boru patlamış olmalı.,Su ve Kanalizasyon
Konteynerin yanına bırakılan eski buzdolabı haftalardır duruyor.,Temizlik ve Çöp
Sokakta dilenciler rahatsız ediyor.,Zabıta
Elektronik atıklarımı nereye bırakabilirim?,Temizlik ve Çöp
Parktaki bankların boyası dökülmüş.,Park ve Bahçeler
Sokaktaki ağacın dalı elektrik tellerine değiyor.,Park ve Bahçeler
Tek yön trafik levhası ters yöne dönmüş.,Ulaşım ve Trafik
Parkın tuvaletleri kilitli duruyor.,Park ve Bahçeler
Eski koltuğumu atmak istiyorum büyük eşya alımı yapıyor musunuz?,Temizlik ve Çöp
Bisiklet yolu ne zaman yapılacak?,Ulaşım ve Trafik
Parkta spor aletlerinden biri çalışmıyor.,Park ve Bahçeler
Fırtınada bir ağaç devrildi.,Park ve Bahçeler
Gece yarısı inşaat çalışması yapılıyor uyuyamıyoruz.,Zabıta
Parktaki salıncak kırılmış çocuklar yaralanabilir.,Park ve Bahçeler
Suyun rengi kahverengi içmek güvenli mi?,Su ve Kanalizasyon
Su kesintisi ne zaman bitecek?,Su ve Kanalizasyon
Köprü üzerindeki trafik her sabah kilitleniyor.,Ulaşım ve Trafik
Mesire alanındaki mangal yerleri yıkılmış.,Park ve Bahçeler
Mahallemizde iki gündür sular akmıyor.,Su ve Kanalizasyon
Metro seferleri gece kaçta bitiyor.,Ulaşım ve Trafik
Suyun basıncı çok düşük üst katlara çıkmıyor.,Su ve Kanalizasyon
Yol çizgileri silinmiş gece hiçbir şey görünmüyor.,Ulaşım ve Trafik
Yasak saatlerde yüksek sesle kaynak yapılıyor.,Zabıta
Sokak yıkama aracı bizim mahalleye hiç gelmiyor.,Temizlik ve Çöp
Yağmurdan sonra bodrum katımızı su bastı.,Su ve Kanalizasyon
Okulun önüne yaya geçidi çizilmesini istiyoruz.,Ulaşım ve Trafik
Parktaki kamelyanın çatısı akıyor.,Park ve Bahçeler
Apartman girişine izinsiz afiş asılmış.,Zabıta
Bahçemizde sürekli ıslak bir alan var altta sızıntı olabilir.,Su ve Kanalizasyon
Otopark ücretleri çok yüksek.,Ulaşım ve Trafik
Pazarda terazisi hileli bir satıcı var.,Zabıta
Pazar yerinden sonra sokak çok kirli kaldı.,Temizlik ve Çöp
Dükkân sahibi masaları tamamen kaldırıma dizmiş.,Zabıta
Kurumuş bir ağaç her an devrilebilir kesilmesi lazım.,Park ve Bahçeler
Geri dönüşüm kutuları hep dolu.,Temizlik ve Çöp
Çocuk oyun alanındaki kum havuzu çok kirli.,Park ve Bahçeler
Kentkartıma bakiye yükledim ama görünmüyor.,Ulaşım ve Trafik"""

df = pd.read_csv(io.StringIO(raw_data))
departmanlar = sorted(df["departman"].unique())
etiketler = df["departman"].map({d: i for i, d in enumerate(departmanlar)}).values

# =============================================================================
# GÖREV 1: TOKENIZATION VE BAĞLAM PENCERESİ KONTROLÜ
# =============================================================================
print("\n" + "="*50 + "\n[GÖREV 1] Tokenization İşlemleri Başlatılıyor...\n" + "="*50)
tokenizer = AutoTokenizer.from_pretrained('openai-community/gpt2')

token_sayilari = [len(tokenizer.encode(m)) for m in df["mesaj"]]
toplam = sum(token_sayilari)

print(f"-> Ortalama Token Sayısı: {np.mean(token_sayilari):.2f}")
print(f"-> En Çok Token'a Bölünen En Uzun Mesaj: {np.max(token_sayilari)}")
print(f"-> 100 Mesajın Toplam Token Sayısı: {toplam}")
print(f"-> GPT-2 Bağlam Pencere Sınırı: {tokenizer.model_max_length}")
print(f"-> Tüm Mesajlar Tek Seferde Pencereye Sığıyor mu?: {toplam <= tokenizer.model_max_length}")

print("\n📝 [GÖREV 1 - TEKNİK YORUM]")
print("Türkçe sondan eklemeli bir dil yapısına sahip olduğu için GPT-2 gibi ağırlıklı olarak İngilizce metinlerle eğitilmiş")
print("Büyük Dil Modellerinin sözlüğünde bütünsel olarak yer almaz. Bu yüzden model Türkçe kelimeleri kök ve eklerine,")
print("hatta bazen karakter seviyesinde hecelere ayırarak böler (Tokenization). Bu durum token sayısını dramatik şekilde artırır.")

# =============================================================================
# GÖREV 2: EMBEDDING VE ANLAM BENZERLİĞİ ANALİZİ
# =============================================================================
print("\n" + "="*50 + "\n[GÖREV 2] Embedding Hesaplaması Başlatılıyor...\n" + "="*50)
model_emb = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")
X = model_emb.encode(df["mesaj"].values)
print(f"-> Matris Çıktı Boyutu: {X.shape} (100 Mesaj, 768 Anlamsal Boyut)")

print("\n📝 [GÖREV 2 - TEKNİK YORUM]")
print("Aynı departmana ait mesajların ortak kelimesi olmasa bile anlamsal kosinüs benzerlik skorları yüksek çıkmaktadır.")
print("Bu durum, kuracağımız sınıflandırıcının kelimelerin kendisine değil, embedding vektörlerinin taşıdığı derin")
print("anlama odaklanarak departmanları birbirinden yüksek başarıyla ayırt edebileceğini (Classification) gösterir.")

# =============================================================================
# GÖREV 3: EĞİTİM/TEST AYRIMI VE SINIFLANDIRICI MODEL YAPISI
# =============================================================================
print("\n" + "="*50 + "\n[GÖREV 3] Yapay Sinir Ağı Eğitimi Başlatılıyor...\n" + "="*50)
X_egitim, X_test, y_egitim, y_test, m_egitim, m_test = train_test_split(
    X, etiketler, df["mesaj"].values, test_size=0.2, stratify=etiketler, random_state=42
)

keras.utils.set_random_seed(42)
siniflandirici = keras.Sequential([
    keras.Input(shape=(768,)),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.Dense(5, activation="softmax")
])

siniflandirici.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])


[INFO] Kütüphaneler kuruluyor, lütfen bekleyin...

[GÖREV 1] Tokenization İşlemleri Başlatılıyor...
-> Ortalama Token Sayısı: 22.95
-> En Çok Token'a Bölünen En Uzun Mesaj: 36
-> 100 Mesajın Toplam Token Sayısı: 2295
-> GPT-2 Bağlam Pencere Sınırı: 1024
-> Tüm Mesajlar Tek Seferde Pencereye Sığıyor mu?: False

📝 [GÖREV 1 - TEKNİK YORUM]
Türkçe sondan eklemeli bir dil yapısına sahip olduğu için GPT-2 gibi ağırlıklı olarak İngilizce metinlerle eğitilmiş
Büyük Dil Modellerinin sözlüğünde bütünsel olarak yer almaz. Bu yüzden model Türkçe kelimeleri kök ve eklerine,
hatta bazen karakter seviyesinde hecelere ayırarak böler (Tokenization). Bu durum token sayısını dramatik şekilde artırır.

[GÖREV 2] Embedding Hesaplaması Başlatılıyor...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

-> Matris Çıktı Boyutu: (100, 768) (100 Mesaj, 768 Anlamsal Boyut)

📝 [GÖREV 2 - TEKNİK YORUM]
Aynı departmana ait mesajların ortak kelimesi olmasa bile anlamsal kosinüs benzerlik skorları yüksek çıkmaktadır.
Bu durum, kuracağımız sınıflandırıcının kelimelerin kendisine değil, embedding vektörlerinin taşıdığı derin
anlama odaklanarak departmanları birbirinden yüksek başarıyla ayırt edebileceğini (Classification) gösterir.

[GÖREV 3] Yapay Sinir Ağı Eğitimi Başlatılıyor...
